# 03 · Herramientas y caso integrador

**Edición docente · 8 de octubre de 2026 · Fundamentos de Business Analytics**

**Núcleo U3 / RA3 · CE 3.1–3.6**
Resolverás el mismo problema con filtros, fórmulas y tablas dinámicas, compararás Python con hojas
de cálculo y SQL, y redactarás una recomendación con un análisis de sensibilidad.

**Cómo trabajar:** ejecuta desde la primera celda. En cada simulador formula primero una predicción,
cambia un parámetro y justifica la diferencia. Los datos son casos docentes del repositorio o
simulaciones identificadas; no permiten inferir resultados de empresas reales.

Los controles requieren JupyterLab con `ipywidgets` y un kernel activo; las salidas guardadas permiten
leer los ejemplos sin ejecutar. Las funciones también pueden llamarse directamente, con lo que
el ejercicio sigue siendo utilizable si el visor no muestra widgets.

[Guía maestra](../guia_maestra_ba.html) · [Manual científico](../material_propio/FBA_manual_cientifico.pdf)

### Antes de ejecutar

Núcleo · 150 min guiados + 60 min autónomos orientativos.

**Objetivo:** Reproducir un indicador con filtros, agregaciones y una unión validada.

**Preparación:** Módulos 01–02; claves de una tabla y lectura de una fórmula.

**Ejemplo de referencia:** Conciliar el ingreso de QuillayMarket entre pandas y SQL sobre las mismas filas.

**Práctica:** Introducir una clave repetida en una copia y detectar por qué se inflan los totales.

**Criterio de logro:** Entregar cálculo conciliado, control de cardinalidad y recomendación con sensibilidad.

**Distribución orientativa:** explicación 30 min; práctica guiada 120 min; trabajo autónomo 60 min. Ajustar tras una clase piloto.

[Guía y secuencia](../guia_maestra_ba.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/FBA_manual_cientifico.html#sec-herramientas)

In [1]:
from pathlib import Path
import sys

RAIZ = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "_transversal" / "datos").is_dir()), None
)
if RAIZ is None:
    raise FileNotFoundError("Abra Jupyter desde el repositorio completo o una subcarpeta.")
CURSO = RAIZ / "01_Fundamentos_Business_Analytics"
sys.path.insert(0, str(CURSO / "reproducibilidad"))
from fba_recursos import ventas, resumen_ventas, pregunta, DATOS, SEMILLA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

%matplotlib inline
plt.rcParams.update(
    {
        "figure.figsize": (8, 4),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.18,
    }
)
pd.set_option("display.max_columns", 14)
df = ventas()
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · {len(df)} transacciones")

Python 3.12.13 · pandas 3.0.6 · 180 transacciones


<a id="herramientas"></a>

## Selección de datos y reglas de negocio

Pregunta: ¿qué productos y regiones merecen una revisión comercial en el segundo trimestre del
caso QuillayMarket? El archivo cubre enero–junio de 2025. La unidad es la transacción; el ingreso suma
`Monto`, el ticket divide ingreso por transacciones y el precio medio por unidad divide ingreso por
unidades. Ninguno equivale a utilidad porque no se dispone de costos observados.

En una hoja de cálculo conviene convertir el rango en tabla y asignar el nombre `Ventas`.
`SUMA`, `PROMEDIO`, `MEDIANA`, `DESVEST.M`, `CONTAR.SI`, `SUMAR.SI.CONJUNTO`, `SI`, `BUSCARX` y
`FILTRAR` resuelven problemas distintos. `BUSCARX` devuelve coincidencias: no sustituye un control
de claves. El separador de argumentos y los nombres de funciones dependen de la configuración.
El [libro propio FBA_herramientas.xlsx](../material_propio/FBA_herramientas.xlsx)
incluye los datos, fórmulas, controles de valores y una práctica de conciliación. Los resultados cacheados se
calculan en Python; recalcula en Excel para contrastar su motor. 

In [2]:
from _transversal.herramientas import crear_libro_fba

libro_fba = crear_libro_fba()
print("Libro propio disponible:", libro_fba.name)

Libro propio disponible: FBA_herramientas.xlsx


In [3]:
q2 = df.loc[df.Fecha.ge("2025-04-01") & df.Fecha.lt("2025-07-01")].copy()
display(pd.Series(resumen_ventas(q2), name="Segundo trimestre"))
equivalencias = pd.DataFrame(
    [
        ["Total", "=SUMA(Ventas[Monto])", "df.Monto.sum()"],
        ["Media", "=PROMEDIO(Ventas[Monto])", "df.Monto.mean()"],
        ["Dispersión muestral", "=DESVEST.M(Ventas[Monto])", "df.Monto.std(ddof=1)"],
        [
            "Condicional",
            '=SI([@Monto]>=100000;"Alto";"Regular")',
            'np.where(df.Monto>=100000,"Alto","Regular")',
        ],
        ["Buscar", "=BUSCARX(clave;claves;valores)", 'merge(..., validate="many_to_one")'],
        ["Agrupar", "Tabla dinámica: Producto; Suma de Monto", 'groupby("Producto").Monto.sum()'],
    ],
    columns=["Operación", "Excel en español (orientativo)", "pandas"],
)
display(equivalencias)
assert not q2.empty and q2.Fecha.dt.quarter.eq(2).all()

transacciones        8.500000e+01
unidades             2.240000e+02
ingreso              8.866376e+07
ticket_medio         1.043103e+06
precio_por_unidad    3.958204e+05
Name: Segundo trimestre, dtype: float64

,Operación,Excel en español (orientativo),pandas
0,Total,=SUMA(Ventas[Monto]),df.Monto.sum()
1,Media,=PROMEDIO(Ventas[Monto]),df.Monto.mean()
2,Dispersión muestral,=DESVEST.M(Ventas[Monto]),df.Monto.std(ddof=1)
3,Condicional,"=SI([@Monto]>=100000;""Alto"";""Regular"")","np.where(df.Monto>=100000,""Alto"",""Regular"")"
4,Buscar,=BUSCARX(clave;claves;valores),"merge(..., validate=""many_to_one"")"
5,Agrupar,Tabla dinámica: Producto; Suma de Monto,"groupby(""Producto"").Monto.sum()"


### Filtros interactivos y selección vacía

El tablero informa explícitamente cuándo la selección no contiene registros. Un grupo vacío no
equivale a un ticket de cero. Cambia región, producto y trimestre; compara valores totales y por
unidad para evitar confundir tamaño con rendimiento.

In [4]:
def filtrar(region="Todas", producto="Todos", trimestre=2):
    filtro = df.Fecha.dt.quarter.eq(trimestre)
    if region != "Todas":
        filtro &= df.Region.eq(region)
    if producto != "Todos":
        filtro &= df.Producto.eq(producto)
    return df.loc[filtro].copy()


def explorar_filtros(region="Todas", producto="Todos", trimestre=2):
    d = filtrar(region, producto, trimestre)
    if d.empty:
        print("Selección sin registros. No se interpreta como ticket cero.")
    display(pd.Series(resumen_ventas(d)))
    display(d.head(8))
    return len(d)


explorar_filtros()
widgets.interact(
    explorar_filtros,
    region=["Todas", *sorted(df.Region.unique())],
    producto=["Todos", *sorted(df.Producto.unique())],
    trimestre=[1, 2],
)

transacciones        8.500000e+01
unidades             2.240000e+02
ingreso              8.866376e+07
ticket_medio         1.043103e+06
precio_por_unidad    3.958204e+05
dtype: float64

,ID_Venta,Fecha,Mes,Region,Producto,Producto_Nombre,Vendedor,Canal,Unidades,Precio_Unitario,Monto
95,1095,2025-04-02,Abril,Norte,A,"Notebook 14""",Ignacio Fuentes,Online,3,476990.0,1430970.0
96,1096,2025-04-02,Abril,Norte,A,"Notebook 14""",Felipe Navarro,Tienda física,3,502990.0,1508970.0
97,1097,2025-04-03,Abril,Sur,C,"Smart TV 55""",Tomás Herrera,Tienda física,2,369990.0,739980.0
98,1098,2025-04-04,Abril,Norte,A,"Notebook 14""",Tomás Herrera,Tienda física,3,502990.0,1508970.0
99,1099,2025-04-04,Abril,Sur,B,Refrigerador 300 L,Tomás Herrera,Tienda física,4,426990.0,1707960.0
100,1100,2025-04-04,Abril,Sur,A,"Notebook 14""",Tomás Herrera,Online,1,502990.0,502990.0
101,1101,2025-04-08,Abril,Norte,C,"Smart TV 55""",Ignacio Fuentes,Online,2,389990.0,779980.0
102,1102,2025-04-08,Abril,Sur,D,Aspiradora robot,Benjamín Araya,Online,1,189990.0,189990.0


interactive(children=(Dropdown(description='region', options=('Todas', 'Centro', 'Norte', 'Sur'), value='Todas…

<function __main__.explorar_filtros(region='Todas', producto='Todos', trimestre=2)>

<a id="tablas-dinamicas"></a>

## Tablas dinámicas y agregaciones

En Excel: insertar tabla dinámica desde `Ventas`, poner `Region` en filas, `Producto` en columnas
y `Monto` en valores como **suma**, no como conteo. Añadir `Fecha` como filtro de trimestre.
Comparar el total general con un cálculo independiente. En pandas `pivot_table` hace la misma
agregación. Para un ticket general se suman los montos y se divide por el total de transacciones:
no se promedian tickets de grupos de tamaños distintos sin ponderación.

In [5]:
tabla = q2.pivot_table(
    index="Region", columns="Producto", values="Monto", aggfunc="sum", fill_value=0
)
display(tabla)
agrupado = q2.groupby("Producto").agg(
    ingreso=("Monto", "sum"), transacciones=("ID_Venta", "size"), unidades=("Unidades", "sum")
)
agrupado["ticket"] = agrupado.ingreso / agrupado.transacciones
agrupado["precio_ponderado"] = agrupado.ingreso / agrupado.unidades
display(agrupado.sort_values("ingreso", ascending=False))
assert np.isclose(tabla.to_numpy().sum(), q2.Monto.sum())
assert np.isclose(np.average(agrupado.ticket, weights=agrupado.transacciones), q2.Monto.mean())

Producto,A,B,C,D
Region,,,,
Centro,3974920.0,9853780.0,6839820.0,5969700.0
Norte,7628850.0,6703850.0,9359760.0,759960.0
Sur,17143670.0,11426740.0,6862820.0,2139890.0


,ingreso,transacciones,unidades,ticket,precio_ponderado
Producto,,,,,
A,28747440.0,21,56,1.368926e+06,513347.142857
B,27984370.0,21,63,1.332589e+06,444196.349206
C,23062400.0,21,60,1.098210e+06,384373.333333
D,8869550.0,22,45,4.031614e+05,197101.111111


<a id="busquedas-sql"></a>

## Búsquedas, uniones y SQL

Una unión muchos-a-uno añade atributos sin multiplicar transacciones. Validar cardinalidad evita
que un catálogo duplicado infle ingresos. La base SQL del ejemplo vive en memoria: no requiere
credenciales, servidor ni conexión a internet. La consulta usa parámetros para los filtros.
La coincidencia entre SQL y pandas es una reconciliación independiente del mismo cálculo.

In [6]:
catalogo = df[["Producto", "Producto_Nombre"]].drop_duplicates()
assert catalogo.Producto.is_unique
base = df.drop(columns="Producto_Nombre").merge(
    catalogo, on="Producto", how="left", validate="many_to_one"
)
assert len(base) == len(df) and np.isclose(base.Monto.sum(), df.Monto.sum())
import sqlite3

with sqlite3.connect(":memory:") as conexion:
    df.assign(Fecha=df.Fecha.dt.strftime("%Y-%m-%d")).to_sql("ventas", conexion, index=False)
    sql = pd.read_sql_query(
        """SELECT Producto, SUM(Monto) AS ingreso
        FROM ventas WHERE Fecha >= ? AND Fecha < ? GROUP BY Producto ORDER BY Producto""",
        conexion,
        params=["2025-04-01", "2025-07-01"],
    )
display(sql)
esperado = q2.groupby("Producto").Monto.sum().sort_index()
assert np.allclose(sql.ingreso, esperado.values)
try:
    df.merge(pd.concat([catalogo, catalogo.head(1)]), on="Producto", validate="many_to_one")
except pd.errors.MergeError:
    print("Una búsqueda con clave duplicada se detecta antes de alterar el total.")

,Producto,ingreso
0,A,28747440.0
1,B,27984370.0
2,C,23062400.0
3,D,8869550.0


Una búsqueda con clave duplicada se detecta antes de alterar el total.


<a id="sensibilidad"></a>

## Sensibilidad económica: una hipótesis no es un dato

Para discutir una campaña se propone un crecimiento de unidades $g$ y descuento proporcional $d$.
Si se mantiene mezcla, no hay restricciones y el costo variable por unidad permanece constante,
un margen base hipotético $m$ implica contribución posterior $R(1+g)(m-d)-C$, frente a $Rm$.
La diferencia es $R[(1+g)(m-d)-m]-C$. No sabemos $m$, el efecto causal de la campaña ni la capacidad;
por ello el simulador explora escenarios y no recomienda una acción como hecho establecido.

In [7]:
def contribucion_incremental(
    ingreso, crecimiento=0.10, descuento=0.05, margen=0.30, costo_fijo=0.0
):
    if (
        ingreso < 0
        or crecimiento < -1
        or not 0 <= descuento < 1
        or not 0 <= margen <= 1
        or costo_fijo < 0
    ):
        raise ValueError("Escenario fuera del dominio económico definido")
    return ingreso * ((1 + crecimiento) * (margen - descuento) - margen) - costo_fijo


def explorar_margen(crecimiento=0.10, descuento=0.05, margen=0.30):
    r = q2.Monto.sum()
    delta = contribucion_incremental(r, crecimiento, descuento, margen)
    print(f"Contribución incremental hipotética: {delta:,.2f} en moneda del caso")
    if margen > descuento:
        print(f"Crecimiento de equilibrio sin costo fijo: {descuento/(margen-descuento):.1%}")
    else:
        print("El descuento consume todo el margen unitario supuesto.")
    return delta


explorar_margen()
widgets.interact(
    explorar_margen,
    crecimiento=(0.0, 0.6, 0.02),
    descuento=(0.0, 0.30, 0.01),
    margen=(0.10, 0.60, 0.02),
)

Contribución incremental hipotética: -2,216,594.00 en moneda del caso
Crecimiento de equilibrio sin costo fijo: 20.0%


interactive(children=(FloatSlider(value=0.1, description='crecimiento', max=0.6, step=0.02), FloatSlider(value…

<function __main__.explorar_margen(crecimiento=0.1, descuento=0.05, margen=0.3)>

<a id="caso-integrador"></a>

## Aplicaciones sectoriales y comunicación

La secuencia pregunta → dato → resumen → decisión se adapta a industria:

| Sector | Decisión | Indicador | Límite que debe declararse |
|---|---|---|---|
| Retail | Revisar surtido | Ingreso, margen, quiebres | Venta no mide demanda no satisfecha |
| Banca | Priorizar revisión de cartera | Tasa de atraso por cohorte | Mezcla y maduración de créditos |
| Logística | Revisar una ruta | Entregas a tiempo / entregas | Distancia, tráfico y composición |
| Salud | Revisar agenda | Espera y ausentismo | Severidad y equidad de acceso |
| SaaS | Investigar abandono | Clientes que salen / base en riesgo | Definición del período y cohorte |

**Producto final:** una página con pregunta, datos, controles de calidad, dos gráficos, dos hallazgos,
una alternativa, supuestos y plan de seguimiento. Se puede proponer un piloto controlado antes
de extender una campaña. El carácter simulado del caso debe aparecer en el informe.

In [8]:
lider = agrupado.ingreso.idxmax()
participacion = agrupado.loc[lider, "ingreso"] / q2.Monto.sum()
mensaje = f"""### Borrador reproducible de informe
**Evidencia:** en el segundo trimestre de este archivo, {lider} aporta {participacion:.1%}
del ingreso registrado; se auditaron {len(q2)} transacciones.
**Interpretación limitada:** concentración de ingreso no demuestra mayor rentabilidad ni causalidad.
**Acción propuesta:** revisar costos, disponibilidad y mezcla; evaluar un piloto si esos datos lo respaldan.
**Seguimiento:** ingreso, contribución y unidades con una comparación adecuada y un responsable definido.
"""
display(Markdown(mensaje))

### Borrador reproducible de informe
**Evidencia:** en el segundo trimestre de este archivo, A aporta 32.4%
del ingreso registrado; se auditaron 85 transacciones.
**Interpretación limitada:** concentración de ingreso no demuestra mayor rentabilidad ni causalidad.
**Acción propuesta:** revisar costos, disponibilidad y mezcla; evaluar un piloto si esos datos lo respaldan.
**Seguimiento:** ingreso, contribución y unidades con una comparación adecuada y un responsable definido.


### Ejercicios y criterios de respuesta

1. Repite el filtro para producto A, región Norte y Q2; si la etiqueta no existe devuelve selección vacía.
   Comprueba el resultado con condiciones conjuntas, no con «o».
2. ¿Por qué no calcular margen como 30% del ingreso sin explicación? Porque el 30% es un supuesto,
   no un campo observado. Debe identificarse como escenario.
3. Con $m=.30$, $d=.05$ y sin costo fijo, el crecimiento de equilibrio es 20%. Con 10% se pierde
   contribución relativa a la base aunque aumenten las unidades.
4. Revisa una propuesta de un compañero usando esta rúbrica formativa: pregunta y grano (20%),
   calidad y reconciliación (25%), análisis y gráficos (25%), decisión y límites (20%), reproducibilidad
   y comunicación (10%). Es una rúbrica de práctica, no reemplaza ponderaciones oficiales.

**Profundización:** diseña qué resultado medirías y cómo asignarías una campaña para estimar su
efecto, en lugar de comparar sin más clientes que recibieron y no recibieron la oferta.

In [9]:
auto_03 = pregunta(
    "El ticket medio general se obtiene…",
    [
        "Promediando cualquier lista de tickets de grupos",
        "Dividiendo ingreso total por transacciones totales",
        "Dividiendo ingreso por unidades",
    ],
    1,
    "La última opción es precio medio ponderado por unidades, un indicador diferente.",
)
assert np.isclose(contribucion_incremental(100, 0.20, 0.05, 0.30), 0)
assert contribucion_incremental(100, 0.10, 0.05, 0.30) < 0
assert len(filtrar("No existe", "Todos", 2)) == 0
print("Comprobaciones del laboratorio 03: OK")

Comprobaciones del laboratorio 03: OK


### Referencias

[pandas: reorganización y tablas dinámicas](https://pandas.pydata.org/docs/user_guide/reshaping.html).
Las equivalencias y escenarios se elaboraron para el archivo QuillayMarket local. El material oficial
de FBA define el alcance de herramientas básicas y comunicación de soluciones.

## Práctica de transferencia 03

Trabaja en una copia del libro FBA_herramientas.xlsx. Filtra un canal y trimestre, calcula ingresos y ticket en Excel y pandas. Duplica una clave del catálogo y explica qué control debe detener una unión; entrega una recomendación de una página.

**Entrega:** hipótesis previa, cálculo con unidades, interpretación y límite. Completa tu respuesta antes de consultar la [pauta docente](../material_propio/PAUTA_DOCENTE.md#nb03).

**Discusión:** ¿Qué control distinguiría una devolución legítima de una copia duplicada?

### Tu resolución

Edita esta celda: escribe tu hipótesis, procedimiento, resultado, interpretación y una comprobación. Adjunta tu código o gráfico si la actividad lo requiere.

**Auto-revisión:** ¿usé la población correcta?, ¿declaré unidades y supuestos?, ¿mi evidencia permite esa conclusión?